# LangGraph RAG Implementation
This notebook implements a standard RAG pipeline using LangGraph, LangChain, and ChromaDB. It uses an LLM-judge to evaluate the performance against the same benchmark used in the PageIndex and General RAG (LlamaIndex) versions.

## Step 1: Set API Key & Imports

In [1]:
import os
from typing import List, TypedDict
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, START, END
from openai import OpenAI
import pandas as pd

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"

## Step 2: Define Benchmark & LLM Judge

In [2]:
benchmark = [
    {
        "question": "Analyze the government's disinvestment performance over the last five years. What is the target for 2025-26, and how does the document characterize the success of achieving these targets in recent years?",
        "expected": "Disinvestment targets have not been achieved for five consecutive years. The target for 2025-26 is Rs 47,000 crore, which is lower than the 2024-25 target of Rs 50,000 crore. In 2024-25, the government is estimated to meet only 66% of its target."
    },
    {
        "question": "Summarize all major initiatives proposed for MSMEs and micro-enterprises in the 2025-26 budget, including credit guarantees, classification changes, and specific financial instruments.",
        "expected": "Key initiatives include: (i) Doubling investment and turnover limits for MSME classification, (ii) Increasing credit guarantee cover to Rs 10 crore for small enterprises and Rs 20 crore for startups and exporters, and (iii) Providing 10 lakh UPI-linked credit cards with a Rs 5 lakh limit for micro-enterprises registered on the Udyam portal."
    },
    {
        "question": "Explain the shift from customs duty to the Agriculture Infrastructure and Development Cess (AIDC) as mentioned in the Finance Bill. What is the stated impact of this shift on the revenue shared with states?",
        "expected": "While the overall tax on items like solar cells and motor vehicles remains similar, there is a shift from customs duty to AIDC cess. This results in a lower proportion of revenue being shared with states because cesses are not part of the shareable tax pool."
    },
    {
        "question": "Describe the new three-year pipeline requirement for infrastructure ministries and the specific maritime and aviation infrastructure missions announced. What are the targets for these missions?",
        "expected": "Infrastructure ministries must formulate a three-year pipeline of PPP projects. A Maritime Development Fund (Rs 25,000 crore corpus, 49% govt contribution) will be set up. A modified UDAN scheme aims to connect 120 new destinations and carry 4 crore passengers in the next 10 years."
    },
    {
        "question": "How does the budget justify the 18.5% increase in allocation for women and children's welfare? Mention the specific linkage to the Pradhan Mantri Awas Yojana and the ownership rules that drive this increase.",
        "expected": "The allocation is Rs 5,65,161 crore. The increase is justified by higher PMAY allocation, as the female head of the family must be the owner or co-owner of the house, thus qualifying the expenditure under women's welfare."
    }
]

def llm_judge(question, expected, predicted):
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key: return 0.0
    client = OpenAI(api_key=api_key)
    prompt = (f"You are an evaluation judge. Compare the predicted answer to the expected answer.\n"
              f"Score from 0 to 100 based on FACTUAL CORRECTNESS only.\n"
              f"Question: {question}\nExpected: {expected}\nPredicted: {predicted}\n"
              f"Respond with ONLY a number.")
    try:
        response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
        score = "".join(c for c in response.choices[0].message.content.strip() if c.isdigit() or c == ".")
        return float(score) if score else 0.0
    except: return 0.0


## Step 3: Initialize Vector Store
Load the PDF, split it into chunks, and store them in ChromaDB.

In [3]:
loader = PyPDFLoader("./budget.pdf")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

## Step 4: Define LangGraph Nodes & State

In [4]:
class State(TypedDict):
    question: str
    context: List[str]
    answer: str

llm = ChatOpenAI(model="gpt-4o", temperature=0)

def retrieve(state: State):
    retrieved_docs = retriever.invoke(state["question"])
    return {"context": [doc.page_content for doc in retrieved_docs]}

def generate(state: State):
    context_text = "\n\n".join(state["context"])
    prompt = f"Use the following context to answer the question: {context_text}\n\nQuestion: {state['question']}"
    response = llm.invoke(prompt)
    return {"answer": response.content}

## Step 5: Build and Compile the Graph

In [5]:
workflow = StateGraph(State)
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)
app = workflow.compile()

## Step 6: Run Benchmark Evaluation

In [6]:
langgraph_results = []

for item in benchmark:
    inputs = {"question": item["question"]}
    result = app.invoke(inputs)
    answer = result["answer"]
    score = llm_judge(item["question"], item["expected"], answer)
    
    langgraph_results.append({
        "Question": item["question"],
        "Expected": item["expected"],
        "Predicted": answer,
        "Score": score
    })
    print(f"Q: {item['question'][:50]}... Score: {score}")

langgraph_df = pd.DataFrame(langgraph_results)
langgraph_accuracy = langgraph_df["Score"].mean()
print(f"\nLangGraph RAG Accuracy: {round(langgraph_accuracy, 2)} %")
langgraph_df.to_csv("langgraph_results.csv", index=False)
langgraph_df

Q: Analyze the government's disinvestment performance... Score: 100.0


Q: Summarize all major initiatives proposed for MSMEs... Score: 95.0


Q: Explain the shift from customs duty to the Agricul... Score: 100.0


Q: Describe the new three-year pipeline requirement f... Score: 100.0


Q: How does the budget justify the 18.5% increase in ... Score: 90.0

LangGraph RAG Accuracy: 97.0 %


,Question,Expected,Predicted,Score
0,Analyze the government's disinvestment perform...,Disinvestment targets have not been achieved f...,"Over the last five years, the government's dis...",100.0
1,Summarize all major initiatives proposed for M...,Key initiatives include: (i) Doubling investme...,"In the 2025-26 budget, several major initiativ...",95.0
2,Explain the shift from customs duty to the Agr...,While the overall tax on items like solar cell...,The Finance Bill outlines a shift from customs...,100.0
3,Describe the new three-year pipeline requireme...,Infrastructure ministries must formulate a thr...,The Union Budget 2025-26 outlines a strategic ...,100.0
4,How does the budget justify the 18.5% increase...,"The allocation is Rs 5,65,161 crore. The incre...",The budget justifies the 18.5% increase in all...,90.0
